In [3]:
import json
import os
import ee
import requests

# Initialize Earth Engine (replace with your GEE cloud project ID if prompted)
ee.Initialize()

os.makedirs("public/data/amazon", exist_ok=True)
os.makedirs("public/data/florida", exist_ok=True)


def export_sar_png(
    collection, region, vis_params, out_path, dimensions=800
):
  """Applies a speckle filter, renders false-color RGB, and downloads a PNG."""
  # Smooth grainy radar speckle with a 30m focal median filter
  img = collection.median().focal_median(30, "circle", "meters").clip(region)
  url = img.getThumbURL({
      "min": vis_params["min"],
      "max": vis_params["max"],
      "bands": vis_params["bands"],
      "dimensions": dimensions,
      "format": "png",
  })
  resp = requests.get(url)
  with open(out_path, "wb") as f:
    f.write(resp.content)
  print(f"Saved: {out_path}")


# ==========================================
# 1. AMAZON DEFORESTATION TIMELAPSE (2024-2026)
# ==========================================
amazon_coords = [-63.35, -9.85, -62.85, -9.35]
amazon_roi = ee.Geometry.Rectangle(amazon_coords)

# False-color RGB: Red = VV (bare soil/roads), Green = VH (forest canopy), Blue = VV - VH
amazon_vis = {"bands": ["VV", "VH", "ratio"], "min": [-20, -25, 0], "max": [-5, -10, 12]}

def add_ratio(img):
  ratio = img.select("VV").subtract(img.select("VH")).rename("ratio")
  return img.addBands(ratio)

s1_amazon = (
    ee.ImageCollection("COPERNICUS/S1_GRD")
    .filterBounds(amazon_roi)
    .filter(ee.Filter.eq("instrumentMode", "IW"))
    .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VV"))
    .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VH"))
    .map(add_ratio)
)

# Export quarterly/bi-monthly frames from 2024 to 2026 (fast & clean for demo)
amazon_periods = [
    ("2024-01-01", "2024-03-01", "2024 Q1"),
    ("2024-05-01", "2024-07-01", "2024 Q2"),
    ("2024-08-01", "2024-10-01", "2024 Q3 (Dry Season Peak)"),
    ("2024-11-01", "2025-01-01", "2024 Q4 (Rainy Season - Cloud Covered)"),
    ("2025-02-01", "2025-04-01", "2025 Q1"),
    ("2025-06-01", "2025-08-01", "2025 Q3"),
    ("2025-10-01", "2025-12-01", "2025 Q4"),
    ("2026-02-01", "2026-05-01", "2026 H1"),
    ("2026-06-01", "2026-10-01", "2026 H2"),
]

amazon_manifest = []
for idx, (start, end, label) in enumerate(amazon_periods):
  sub = s1_amazon.filterDate(start, end)
  fname = f"amazon_{idx:02d}.png"
  export_sar_png(sub, amazon_roi, amazon_vis, f"public/data/amazon/{fname}")
  amazon_manifest.append({
      "id": idx,
      "label": label,
      "startDate": start,
      "endDate": end,
      "image": f"/data/amazon/{fname}",
      # Leaflet expects [[south, west], [north, east]]
      "bounds": [[amazon_coords[1], amazon_coords[0]], [amazon_coords[3], amazon_coords[2]]],
  })

# ==========================================
# 2. FLORIDA HURRICANES HELENE & MILTON (2024)
# ==========================================
florida_coords = [-82.85, 27.65, -82.35, 28.05]  # Tampa Bay
florida_roi = ee.Geometry.Rectangle(florida_coords)
florida_vis = {"bands": ["VV", "VH", "VV"], "min": [-22, -28, -22], "max": [-4, -10, -4]}

s1_florida = (
    ee.ImageCollection("COPERNICUS/S1_GRD")
    .filterBounds(florida_roi)
    .filter(ee.Filter.eq("instrumentMode", "IW"))
)

florida_events = [
    ("2024-09-01", "2024-09-20", "Pre-Storm Baseline (Sept 2024)"),
    ("2024-09-25", "2024-09-28", "Hurricane Helene Surge (Sept 26, 2024)"),
    ("2024-10-01", "2024-10-08", "Between Storms (Early Oct 2024)"),
    ("2024-10-09", "2024-10-14", "Hurricane Milton Landfall (Oct 10-11, 2024)"),
]

florida_manifest = []
for idx, (start, end, label) in enumerate(florida_events):
  sub = s1_florida.filterDate(start, end)
  fname = f"florida_{idx:02d}.png"
  export_sar_png(sub, florida_roi, florida_vis, f"public/data/florida/{fname}")
  florida_manifest.append({
      "id": idx,
      "label": label,
      "startDate": start,
      "endDate": end,
      "image": f"/data/florida/{fname}",
      "bounds": [[florida_coords[1], florida_coords[0]], [florida_coords[3], florida_coords[2]]],
  })

with open("public/data/manifest.json", "w") as f:
  json.dump({"amazon": amazon_manifest, "florida": florida_manifest}, f, indent=2)

print("Done! Manifest and PNGs saved to public/data/")

Saved: public/data/amazon/amazon_00.png
Saved: public/data/amazon/amazon_01.png
Saved: public/data/amazon/amazon_02.png
Saved: public/data/amazon/amazon_03.png
Saved: public/data/amazon/amazon_04.png
Saved: public/data/amazon/amazon_05.png
Saved: public/data/amazon/amazon_06.png
Saved: public/data/amazon/amazon_07.png
Saved: public/data/amazon/amazon_08.png
Saved: public/data/florida/florida_00.png
Saved: public/data/florida/florida_01.png
Saved: public/data/florida/florida_02.png
Saved: public/data/florida/florida_03.png
Done! Manifest and PNGs saved to public/data/


In [1]:
import ee
ee.Authenticate()


Successfully saved authorization token.
